_News Summarization using Transformers_

1. Fine tune a transformer based model to summarize descriptive news.
    - Explore and learn about different variants of transformer based models and tasks for which they can be used.
    - List out the models, max token inputs, no. of parameters.
    - Select a suitable variant of BART model, which can be finetuned on free compute platforms like colab/kaggle. 
    - Use the dataset for summarizing the news articles, fine tune the model after loading pretrained a model from hugging face.
    - Explore and learn about different metrics to evaluate the performance of the model for text summarization tasks.
    - Use suitable metrics to evaluate the fine tuned model performance on the test set.

1. Dataset:
    - Indian Language Summarization Dataset 
    - https://huggingface.co/datasets/ILSUM/ILSUM-1.0
    - For this assignment use the “English” subset.
    - Filter subset of dataset such that not much information is lost (because input token size is fixed) and no of training samples > 1000 for fine tuning.
    - Use the same filtering criteria for test and validation set.

In [2]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==

In [7]:
from datasets import load_dataset
import transformers
import torch

In [8]:
dataset=load_dataset("ILSUM/ILSUM-1.0","English")

README.md:   0%|          | 0.00/4.78k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/46.5M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/3.37M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12565 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4487 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/898 [00:00<?, ? examples/s]

In [9]:
print(dataset)
print(dataset["train"])
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 12565
    })
    test: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 4487
    })
    validation: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 898
    })
})
Dataset({
    features: ['id', 'Article', 'Heading', 'Summary'],
    num_rows: 12565
})
{'id': '3938f547c863630032649c54e611e6b0', 'Article': 'Logos for MasterCard and Visa credit cards at the entrance of a New York coffee shopIn the latest blow to Russia’s financial system after its invasion of Ukraine, Mastercard and Visa said they are suspending their operations in the country. Mastercard said cards issued by Russian banks will no longer be supported by its network and any Mastercard issued outside the country will not work at Russian stores or ATMs.“We don’t take this decision lightly,” Mastercard said in a statement, adding that it made

In [10]:
from transformers import AutoTokenizer

tokenizer=transformers.AutoTokenizer.from_pretrained("facebook/bart-base")
max_length=1024

def filter_text(text):
    return len(tokenizer(text['Article'])['input_ids'])<=max_length

filtered_dataset=dataset.filter(filter_text)
print(filtered_dataset)

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Filter:   0%|          | 0/12565 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4487 [00:00<?, ? examples/s]

Filter:   0%|          | 0/898 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 9781
    })
    test: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 3441
    })
    validation: Dataset({
        features: ['id', 'Article', 'Heading', 'Summary'],
        num_rows: 694
    })
})


In [11]:
from transformers import BartForConditionalGeneration,BartTokenizer 


model=BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

tokenizer=BartTokenizer.from_pretrained("facebook/bart-base")

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [12]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                                  Param #
BartForConditionalGeneration                            --
├─BartModel: 1-1                                        --
│    └─BartScaledWordEmbedding: 2-1                     38,603,520
│    └─BartEncoder: 2-2                                 --
│    │    └─BartScaledWordEmbedding: 3-1                38,603,520
│    │    └─BartLearnedPositionalEmbedding: 3-2         787,968
│    │    └─ModuleList: 3-3                             42,527,232
│    │    └─LayerNorm: 3-4                              1,536
│    └─BartDecoder: 2-3                                 --
│    │    └─BartScaledWordEmbedding: 3-5                38,603,520
│    │    └─BartLearnedPositionalEmbedding: 3-6         787,968
│    │    └─ModuleList: 3-7                             56,710,656
│    │    └─LayerNorm: 3-8                              1,536
├─Linear: 1-2                                           38,603,520
Total params: 255,230,976
Trainable params: 25

In [13]:
def preprocess(text):
    inputs=tokenizer(text['Article'],max_length=1024,truncation=True,padding="max_length")
    outputs=tokenizer(text['Summary'],max_length=128,truncation=True,padding="max_length")
    inputs["labels"]=outputs["input_ids"]
    return inputs

tokenized_dataset=filtered_dataset.map(preprocess,batched=True)

Map:   0%|          | 0/9781 [00:00<?, ? examples/s]

Map:   0%|          | 0/3441 [00:00<?, ? examples/s]

Map:   0%|          | 0/694 [00:00<?, ? examples/s]

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./outputs',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    do_eval=True,
    report_to=[]
  )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer
)

/tmp/ipykernel_31/1347804233.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.537700,0.096201
2,0.096300,0.095580
3,0.081200,0.094687


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=1836, training_loss=0.20805075901006562, metrics={'train_runtime': 3166.44, 'train_samples_per_second': 9.267, 'train_steps_per_second': 0.58, 'total_flos': 1.789149689413632e+16, 'train_loss': 0.20805075901006562, 'epoch': 3.0})

In [16]:
import evaluate

rougue = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=decoded_preds, references=decoded_labels)

In [17]:
trainer.compute_metrics = compute_metrics

trainer.evaluate(eval_dataset=tokenized_dataset["test"], metric_key_prefix="test")

{'test_loss': 0.08755794167518616,
 'test_runtime': 88.1545,
 'test_samples_per_second': 39.034,
 'test_steps_per_second': 0.613,
 'epoch': 3.0}

In [18]:
example=dataset["test"][0]
display(example['Article'])

'Fear shakes Mexico border city after violence leaves 18 deadFear has invaded the Mexican border city of Reynosa after gunmen in vehicles killed 14 people, including taxi drivers, workers and a nursing student, and security forces responded with operations that left four suspects dead.While this city across the border from McAllen, Texas is used to cartel violence as a key trafficking point, the 14 victims in Saturday’s attacks appeared to be what Tamaulipas Gov. Francisco García Cabeza de Vaca called “innocent citizens” rather than members of one gang killed by a rival.Local businessman Misael Chavarria Garza said many businesses closed early Saturday after the attacks and people were very scared as helicopters flew overhead. On Sunday, he said, “the people were quiet as if nothing had happened, but with a feeling of anger because now crime has happened to innocent people.”“It’s not fair,” said taxi driver Rene Guevara, adding that among the dead were two of his fellow taxi drivers wh

In [19]:
inputs = tokenizer(
    example["Article"],
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

In [21]:
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=128,
    num_beams=4,
    early_stopping=True
)

In [22]:
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n📝 Generated Summary:\n", summary)
print("\n📌 Reference Summary:\n", example["Summary"])


📝 Generated Summary:
 Fear has invaded the Mexican border city of Reynosa after gunmen in vehicles killed 14 people, including taxi drivers, workers and a nursing student, and security forces responded with operations that left four suspects dead.

📌 Reference Summary:
 The attacks took place in several neighborhoods in eastern Reynosa, according to the Tamaulipas state agency that coordinates security forces, and sparked a deployment of the military, National Guard and state police across the city. Images posted on social media showed bodies in the streets.
